# 交易台 Phase 0 数据质量验证

## tl;dr

- 本次验证覆盖 2026-07-10、2026-07-13、2026-07-14 三个交易日，结果为 `conditional_pass`。
- 自动化质量规则通过 46 项、警告 2 项、失败 0 项。
- 上交所 ETF、沪深两融和港交所互联互通均完成三日验证；深交所 ETF 只有无统计日期的当前快照，因此只能从正式运行日起向前积累。

## Context & Methods

目标是判断官方公开源是否足以支撑个人日频资金流交易台。粒度分别是 ETF×交易日、市场×交易日，以及互联互通通道×交易日。所有金额归一到基础货币单位，所有数量归一到股/份。原始响应与 SHA-256 哈希由采集器保存。

### Key Assumptions

- 港交所北向在 2024-08-19 后不再披露买卖额，因此北向净流入必须为空。
- 深交所两融页面单位为亿元/亿股，标准化时乘以 1e8。
- 深交所 ETF 当前下载没有源站统计日期，采集日期只能作为推断日期并产生高等级警告。

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
quality = json.loads((ROOT / 'reports/phase0/latest-quality-summary.json').read_text(encoding='utf-8'))
manifest = json.loads((ROOT / 'reports/phase0/latest-manifest.json').read_text(encoding='utf-8'))

def load_dataset(name: str) -> pd.DataFrame:
    return pd.read_csv(ROOT / manifest['combined_paths'][name])

quality['phase0_gate_status'], manifest['run_id']

('conditional_pass', '20260716T003352+0800')

## Data

本节检查每个规范化数据集的行数、日期覆盖和缺失模式。

In [2]:
profile_rows = []
for dataset, item in quality['profiles'].items():
    profile_rows.append({
        'dataset': dataset,
        'rows': item['rows'],
        'dates': ', '.join(item['distinct_trade_dates']),
        'date_count': len(item['distinct_trade_dates']),
    })
pd.DataFrame(profile_rows).sort_values('dataset').reset_index(drop=True)

,dataset,rows,dates,date_count
0,sse_etf,2634,"2026-07-10, 2026-07-13, 2026-07-14",3
1,sse_margin,3,"2026-07-10, 2026-07-13, 2026-07-14",3
2,stock_connect_summary,12,"2026-07-10, 2026-07-13, 2026-07-14",3
3,stock_connect_top,120,"2026-07-10, 2026-07-13, 2026-07-14",3
4,szse_etf,1007,2026-07-15,1
5,szse_margin,3,"2026-07-10, 2026-07-13, 2026-07-14",3


## Results

质量门槛没有失败项；需要人工理解的证据集中在深交所 ETF 日期语义。

In [3]:
non_pass = pd.DataFrame([item for item in quality['findings'] if item['status'] != 'pass'])
non_pass[['dataset', 'status', 'severity', 'message', 'remediation']]

,dataset,status,severity,message,remediation
0,szse_etf,warn,high,统计日期由采集任务推断,只从当日定时快照向前积累，不把推断日期用于历史回填。
1,szse_etf,warn,high,未覆盖全部请求日期,深交所只提供当前快照；从正式上线日起每日留存，不宣称可官方回填。


### 两融单位与跨市场汇总

In [4]:
margin = pd.concat([load_dataset('sse_margin'), load_dataset('szse_margin')], ignore_index=True)
margin_daily = margin.groupby('trade_date', as_index=False).agg(
    financing_balance_yuan=('financing_balance_yuan', 'sum'),
    financing_buy_yuan=('financing_buy_yuan', 'sum'),
    securities_lending_balance_yuan=('securities_lending_balance_yuan', 'sum'),
    margin_total_balance_yuan=('margin_total_balance_yuan', 'sum'),
)
margin_daily['financing_balance_billion_yuan'] = margin_daily['financing_balance_yuan'] / 1e9
margin_daily['financing_balance_change_billion_yuan'] = margin_daily['financing_balance_yuan'].diff() / 1e9
margin_daily[['trade_date', 'financing_balance_billion_yuan', 'financing_balance_change_billion_yuan']]

,trade_date,financing_balance_billion_yuan,financing_balance_change_billion_yuan
0,2026-07-10,2913.768507,NaN
1,2026-07-13,2880.236049,-33.532458
2,2026-07-14,2869.278186,-10.957863


### 互联互通只使用仍被官方披露的口径

In [5]:
connect = load_dataset('stock_connect_summary')
south = connect[connect['direction'] == 'southbound'].groupby('trade_date', as_index=False)['net_buy'].sum()
south['southbound_net_buy_billion_hkd'] = south['net_buy'] / 1e9
north = connect[connect['direction'] == 'northbound'].groupby('trade_date', as_index=False)['total_turnover'].sum()
north['northbound_turnover_billion_rmb'] = north['total_turnover'] / 1e9
south[['trade_date', 'southbound_net_buy_billion_hkd']].merge(
    north[['trade_date', 'northbound_turnover_billion_rmb']], on='trade_date'
)

,trade_date,southbound_net_buy_billion_hkd,northbound_turnover_billion_rmb
0,2026-07-10,-5.99454,449.28034
1,2026-07-13,9.03842,417.81063
2,2026-07-14,10.99688,415.90437


### ETF 日期覆盖

In [6]:
sse_etf = load_dataset('sse_etf')
szse_etf = load_dataset('szse_etf')
pd.concat([
    sse_etf.groupby(['market', 'trade_date'], as_index=False).agg(rows=('instrument_code', 'size'), total_shares=('shares', 'sum')),
    szse_etf.groupby(['market', 'trade_date'], as_index=False).agg(rows=('instrument_code', 'size'), total_shares=('shares', 'sum')),
], ignore_index=True)

,market,trade_date,rows,total_shares
0,SSE,2026-07-10,878,1.917991e+12
1,SSE,2026-07-13,878,1.938573e+12
2,SSE,2026-07-14,878,1.946562e+12
3,SZSE,2026-07-15,1007,1.160332e+12


## Takeaways

1. 可以进入 Phase 1 的接口和数据模型开发：上交所 ETF、沪深两融和港交所互联互通均具备可重放的官方数据链路。
2. 深交所 ETF 历史回填不能依赖当前官方接口；上线后必须每天保存快照，并在累计三个真实快照前保持 `conditional_pass`。
3. 只有三个观测日，不足以绘制有解释力的趋势图；本 Notebook 保留精确表格，趋势分析留到 20 个交易日后。